# Lab 6: Exploratory Data Analysis with pandas

## Goal

- Load and explore historical transaction data,
- Feature engineering for fraud detection,
- Visualize patterns that inform our streaming rules.

**Business case (continued):** Before building better models, we need to understand our data. This lab uses historical (batch) data to discover patterns we'll apply in streaming.

---

## Part 1: Load and Explore

### Task 1.1 -- Generate a realistic dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 5000

df = pd.DataFrame({
    'tx_id': [f'TX{str(i).zfill(5)}' for i in range(n)],
    'user_id': [f'u{np.random.randint(1,100):02d}' for _ in range(n)],
    'amount': np.random.lognormal(5, 1.2, n).clip(5, 15000).round(2),
    'store': np.random.choice(['Warsaw','Krakow','Gdansk','Wroclaw'], n, p=[0.4,0.25,0.2,0.15]),
    'category': np.random.choice(['electronics','clothing','food','books','home'], n),
    'hour': np.random.choice(range(24), n, p=[0.01]*6 + [0.05]*4 + [0.08]*4 + [0.06]*4 + [0.04]*4 + [0.02]*2),
    'day_of_week': np.random.choice(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], n),
    'fraud': 0
})

# Inject fraud patterns
fraud_idx = df.sample(frac=0.04, random_state=42).index
df.loc[fraud_idx, 'fraud'] = 1
df.loc[fraud_idx, 'amount'] = np.random.uniform(2000, 12000, len(fraud_idx))
df.loc[fraud_idx, 'hour'] = np.random.choice([0,1,2,3,4,23], len(fraud_idx))

print(f"Shape: {df.shape}, Fraud rate: {df['fraud'].mean():.1%}")
df.head()

### Task 1.2 -- Basic statistics

1. What is the mean, median, std of `amount` for fraud vs non-fraud?
2. Which store has the highest fraud rate?
3. What hours have the most fraud?

In [ ]:
# YOUR CODE
# 1. df.groupby('fraud')['amount'].describe()
# 2. Fraud rate per store
# 3. Fraud count per hour

### Task 1.3 -- Visualizations

1. Histogram of `amount` (fraud vs non-fraud, overlapping)
2. Bar chart: fraud rate per store
3. Heatmap: fraud rate by hour and day_of_week

In [ ]:
# YOUR CODE

---

## Part 2: Feature Engineering

### Task 2.1 -- Create features for modeling

Add columns that could help detect fraud:
- `log_amount`: log of amount
- `is_night`: 1 if hour < 6 or hour >= 22
- `is_high_value`: 1 if amount > 95th percentile
- `user_tx_count`: number of transactions per user (use transform)
- `user_avg_amount`: average amount per user

In [ ]:
# YOUR CODE

### Task 2.2 -- Correlation analysis

Compute correlation of all numeric features with `fraud`. Which features are most predictive?

In [ ]:
# YOUR CODE

---

## Part 3: Train a Better Model

### Task 3.1 -- Train and evaluate

Using the engineered features, train a RandomForest and compare with a simple model (amount only).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# YOUR CODE
# 1. Define feature sets: simple = ['amount'] vs full = all engineered features
# 2. Train both models
# 3. Compare classification_report
# 4. Save the better model to 'fraud_model_v2.pkl'

---

## Homework

1. Find 2 more features that improve detection (be creative!).
2. Export your feature engineering as a function -- we'll reuse it in streaming.
3. Push to Git.

**Next lab:** Same analysis but in Apache Spark -- working at scale.